24F-0040
laiba
Assignment 02
Artificial Intelligence
BSAI(4A)

In [15]:
import tkinter as tk  # for GUI
from tkinter import messagebox  # for pop up message
import heapq  # for priority queue(for A* search)
import random  # for random obstacles
import math  # for measuring distances
import time

cell_size = 25
rows = 20
cols = 20
grid = []
start = (0, 0)
goal = (19, 19)
canvas = None
mat_lbl = None
algo = None
hrr_type = None
dyn_mode = None

def create_grid():
    global grid, start, goal
    grid = [[0 for _ in range(cols)] for _ in range(rows)]
    start = (0, 0)
    goal = (rows - 1, cols - 1)
    canvas.delete("all")
    drawfullgrid()

def draw_cell(r, c, clr):
    x1 = c * cell_size
    y1 = r * cell_size
    x2 = x1 + cell_size
    y2 = y1 + cell_size
    canvas.create_rectangle(x1, y1, x2, y2, fill=clr, outline="red")
    
def drawfullgrid():
    for r in range(rows):
        for c in range(cols):
            color = "orchid"
            if (r, c) == start:
                color = "beige"
            elif (r, c) == goal:
                color = "peru"
            elif grid[r][c] == 1:
                color = "teal"
            draw_cell(r, c, color)

def toggle_obstacle(evnt):
    c = evnt.x // cell_size
    r = evnt.y // cell_size
    if 0 <= r < rows and 0 <= c < cols:
        if (r, c) not in [start, goal]:
            grid[r][c] = 1 if grid[r][c] == 0 else 0
            drawfullgrid()

def random_map():
    try:
        density = int(density_entry.get()) / 100
        for r in range(rows):
            for c in range(cols):
                if (r, c) not in [start, goal]:
                    grid[r][c] = 1 if random.random() < density else 0
        drawfullgrid()
    except:
        messagebox.showerror("Error", "You entered invalid value")

def heuristic(a, b):
    if heuristic_type.get() == "Manhattan":
        return abs(a[0] - b[0]) + abs(a[1] - b[1])
    else:
        return math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)
        
def get_neighbors(pos):
    r, c = pos
    dirc = [(1, 0), (-1, 0), (0, 1), (0, -1)]
    nbr = []
    for dr, dc in dirc:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols:
            if grid[nr][nc] == 0:
                nbr.append((nr, nc))
    return nbr

def run_search():
    global start
    open_list = []
    heapq.heappush(open_list, (0, start))
    g_cost = {start: 0}
    parent = {}
    closed = set()
    vis = 0
    start_time = time.time()
    while open_list:
        currf, curr = heapq.heappop(open_list)
        if curr in closed:
            continue
        closed.add(curr)
        vis += 1
        if curr != start:
            draw_cell(curr[0], curr[1], "lavender")
            root.update()
        if curr == goal:
            end_time = time.time()
            exe_time = int((end_time - start_time) * 1000)
            path = rec_path(parent)
            move_agent(path, vis, exe_time)
            return
        for nb in get_neighbors(curr):
            tnt_g = g_cost[curr] + 1
            if nb not in g_cost or tnt_g < g_cost[nb]:
                g_cost[nb] = tnt_g
                h = heuristic(nb, goal)
                if algorithm.get() == "Greedy":
                    f = h
                else:
                    f = tnt_g + h
                heapq.heappush(open_list, (f, nb))
                parent[nb] = curr
                if nb != goal:
                    draw_cell(nb[0], nb[1], "black")
                    root.update()
    messagebox.showinfo("Result", "No Path Found")
            
def rec_path(parent):
    path = []
    node = goal
    while node in parent:
        path.append(node)
        node = parent[node]
    path.append(start)
    path.reverse()
    return path

def spawn_dynamic_obstacle():
    if random.random() < 0.1:
        r = random.randint(0, rows - 1)
        c = random.randint(0, cols - 1)
        if (r, c) not in [start, goal]:
            grid[r][c] = 1
            draw_cell(r, c, "purple")

def move_agent(path, nodes_visited, execution_time):
    global start
    curr = start
    steps = path[1:]
    
    def step_move(idx):
        nonlocal curr
        if idx >= len(steps):
            cost = len(path) - 1
            metrics_label.config(text=f"Nodes: {nodes_visited} | Cost: {cost} | Time: {execution_time} ms")
            return
        step = steps[idx]
        if dynamic_mode.get():
            spawn_dynamic_obstacle()
        if grid[step[0]][step[1]] == 1:
            messagebox.showinfo("Replanning", "Path blocked! plan another way")
            start = curr
            run_search()
            return
        draw_cell(curr[0], curr[1], "pink")
        draw_cell(step[0], step[1], "green")
        curr = step
        root.update()
        root.after(200, lambda: step_move(idx + 1))

    step_move(0)

def update_grid_size():
    global rows, cols
    try:
        rows = int(rows_entry.get())
        cols = int(cols_entry.get())
        canvas.config(width=cols * cell_size, height=rows * cell_size)
        create_grid()
    except:
        messagebox.showerror("Error", "you entered Invalid grid size")

def reset_grid():
    create_grid()
    metrics_label.config(text="Nodes: 0 | Cost: 0 | Time: 0 ms")

try:
    root.destroy()
except:
    pass

root = tk.Tk()
root.title("Dynamic Pathfinding Agent")
frame = tk.Frame(root)
frame.pack()
tk.Label(frame, text="Rows:").pack(side=tk.LEFT)
rows_entry = tk.Entry(frame, width=5)
rows_entry.insert(0, "20")
rows_entry.pack(side=tk.LEFT)
tk.Label(frame, text="Cols:").pack(side=tk.LEFT)
cols_entry = tk.Entry(frame, width=5)
cols_entry.insert(0, "20")
cols_entry.pack(side=tk.LEFT)
tk.Button(frame, text="Create Grid", command=update_grid_size).pack(side=tk.LEFT)
tk.Label(frame, text="Obstacle %:").pack(side=tk.LEFT)
density_entry = tk.Entry(frame, width=5)
density_entry.insert(0, "30")
density_entry.pack(side=tk.LEFT)
tk.Button(frame, text="Random Map", command=random_map).pack(side=tk.LEFT)
algorithm = tk.StringVar(value="A*")
tk.OptionMenu(frame, algorithm, "A*", "Greedy").pack(side=tk.LEFT)
heuristic_type = tk.StringVar(value="Manhattan")
tk.OptionMenu(frame, heuristic_type, "Manhattan", "Euclidean").pack(side=tk.LEFT)
dynamic_mode = tk.BooleanVar()
tk.Checkbutton(frame, text="Dynamic Mode", variable=dynamic_mode).pack(side=tk.LEFT)
tk.Button(frame, text="Run Search", command=run_search).pack(side=tk.LEFT)
tk.Button(frame, text="Reset", command=reset_grid).pack(side=tk.LEFT)
metrics_label = tk.Label(root, text="Nodes: 0 | Cost: 0 | Time: 0 ms")
metrics_label.pack()
canvas = tk.Canvas(root, width=cols * cell_size, height=rows * cell_size)
canvas.pack()
canvas.bind("<Button-1>", toggle_obstacle)
create_grid()
root.mainloop()